In [17]:
import pandas as pd
import numpy as np
from pathlib import Path

projectDir = Path.cwd().resolve().parent
dataPath = projectDir / "data" / "processed" / "matchupDiff.csv"

assert dataPath.exists(), f"Missing file: {dataPath}"

matchupDiff = pd.read_csv(dataPath)

In [18]:
rawPath = projectDir / "data" / "raw" / "MarchMadnessGameStats2003-2024.csv"
assert rawPath.exists(), f"Missing file: {rawPath}"

gameStats = pd.read_csv(rawPath)

winnerBox = gameStats[["Season", "WTeamID", "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WAst", "WTO"]].copy()
winnerBox.columns = ["Season", "TeamID", "FGM", "FGA", "FGM3", "FGA3", "FTM", "FTA", "AST", "TO"]

loserBox = gameStats[["Season", "LTeamID", "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LAst", "LTO"]].copy()
loserBox.columns = ["Season", "TeamID", "FGM", "FGA", "FGM3", "FGA3", "FTM", "FTA", "AST", "TO"]

teamTotals = pd.concat([winnerBox, loserBox], ignore_index=True)
teamTotals = teamTotals.groupby(["Season", "TeamID"], as_index=False)[["FGM", "FGA", "FGM3", "FGA3", "FTM", "FTA", "AST", "TO"]].sum()

teamTotals["fgPct"] = np.where(teamTotals["FGA"] == 0, 0, (teamTotals["FGM"] / teamTotals["FGA"]) * 100)
teamTotals["threePtPct"] = np.where(teamTotals["FGA3"] == 0, 0, (teamTotals["FGM3"] / teamTotals["FGA3"]) * 100)
teamTotals["ftPct"] = np.where(teamTotals["FTA"] == 0, 0, (teamTotals["FTM"] / teamTotals["FTA"]) * 100)
teamTotals["eFgPct"] = np.where(teamTotals["FGA"] == 0, 0, ((teamTotals["FGM"] + 0.5 * teamTotals["FGM3"]) / teamTotals["FGA"]) * 100)
teamTotals["astTo"] = np.where(teamTotals["TO"] == 0, 0, teamTotals["AST"] / teamTotals["TO"])

teamTotals[["fgPct", "threePtPct", "ftPct", "eFgPct", "astTo"]] = teamTotals[["fgPct", "threePtPct", "ftPct", "eFgPct", "astTo"]].round(1)

teamRates = teamTotals[["Season", "TeamID", "fgPct", "threePtPct", "ftPct", "eFgPct", "astTo"]].copy()

In [19]:
teamOneRates = teamRates.rename(columns={
    "TeamID": "team1Id",
    "fgPct": "teamFgPct",
    "threePtPct": "teamThreePtPct",
    "ftPct": "teamFtPct",
    "eFgPct": "teamEFgPct",
    "astTo": "teamAstTo"
})

teamTwoRates = teamRates.rename(columns={
    "TeamID": "team2Id",
    "fgPct": "oppFgPct",
    "threePtPct": "oppThreePtPct",
    "ftPct": "oppFtPct",
    "eFgPct": "oppEFgPct",
    "astTo": "oppAstTo"
})

In [20]:
matchupDiff = matchupDiff.merge(teamOneRates, on=["Season", "team1Id"], how="left")
matchupDiff = matchupDiff.merge(teamTwoRates, on=["Season", "team2Id"], how="left")

matchupDiff["fgPctDiff"] = (matchupDiff["teamFgPct"] - matchupDiff["oppFgPct"]).round(1)
matchupDiff["threePtPctDiff"] = (matchupDiff["teamThreePtPct"] - matchupDiff["oppThreePtPct"]).round(1)
matchupDiff["ftPctDiff"] = (matchupDiff["teamFtPct"] - matchupDiff["oppFtPct"]).round(1)
matchupDiff["eFgPctDiff"] = (matchupDiff["teamEFgPct"] - matchupDiff["oppEFgPct"]).round(1)
matchupDiff["astToDiff"] = (matchupDiff["teamAstTo"] - matchupDiff["oppAstTo"]).round(1)

matchupDiff[["fgPctDiff", "threePtPctDiff", "ftPctDiff", "eFgPctDiff", "astToDiff"]].head()

,fgPctDiff,threePtPctDiff,ftPctDiff,eFgPctDiff,astToDiff
0,1.3,6.7,-25.9,4.1,0.1
1,14.6,12.3,-22.1,17.1,0.7
2,9.9,9.5,4.4,8.7,0.5
3,0.5,4.6,1.2,-0.3,-0.4
4,2.0,-9.6,-15.0,-1.0,-0.1


In [ ]:
outPath = projectDir / "data" / "processed" / "matchupDiff_week5_features.csv"
matchupDiff.to_csv(outPath, index=False)

Saved: C:\Users\Colin\Desktop\Github\BC-AppliedAnalyticsProject\data\processed\matchupDiff_week5_features.csv
